# DS Quest practice: Python for Data Science

**Fill-in-the-blank version:** each cell starts with a skeleton; replace every `___`. (The blank-page version is `python-blank.ipynb`.)

Practical questions from the **Python for Data Science** module. Write your answer in the code cell under each question, run it, then run the **check** cell right below it. The check runs *in this notebook*, on your own variables, and tells you exactly what is right or wrong.

1. Run the first code cell once (it loads the checker and the 16 questions).
2. Work through the questions in any order.
3. `dsq.hint("question-id")` gives a hint; `dsq.solution("question-id")` shows the reference answer (only after you tried 3 times).
4. `dsq.score()` shows where you stand.

Optional: `dsq.connect("http://127.0.0.1:5177", "YOUR-KEY")` (key from the module page in DS Quest) sends solved questions to the app so your profile counts them.

You need Python with `numpy`, `pandas`, `matplotlib` and `seaborn`, e.g. `pip install numpy pandas matplotlib seaborn notebook`.

In [ ]:
# --- DS Quest practice checker (runs in YOUR notebook, on YOUR variables) ---
import json as _json, sys as _sys, urllib.request as _ur

class _DSQ:
    def __init__(self):
        self.qs = {}; self.order = []; self.tries = {}; self.ok = {}; self.url = None; self.key = None; self.module = ''

    def load(self, pack):
        self.module = pack['title']
        for q in pack['questions']:
            self.qs[q['id']] = q; self.order.append(q['id'])
        print(f"DS Quest: {len(self.order)} questions loaded for '{self.module}'. Write your answer, then run the check cell under it.")
        if self._auto(): print('Linked to your DS Quest account: solved questions count in the app.')

    def connect(self, url, key, quiet=False):
        self.url = url.rstrip('/'); self.key = key
        if not quiet: print('Connected: solved questions will be reported to', self.url)

    def _auto(self):
        # DS Quest writes dsq_config.json next to the notebooks when you open them from the app
        import os
        d = os.getcwd()
        for _ in range(4):
            p = os.path.join(d, 'dsq_config.json')
            if os.path.exists(p):
                try:
                    c = _json.load(open(p)); self.connect(c['url'], c['key'], quiet=True); return True
                except Exception:
                    pass
            d = os.path.dirname(d)
        return False

    def _report(self, qid, ok):
        if not self.url: return
        try:
            req = _ur.Request(self.url + '/api/nb/report', data=_json.dumps({'key': self.key, 'qid': qid, 'ok': bool(ok)}).encode(), headers={'Content-Type': 'application/json'})
            _ur.urlopen(req, timeout=4).read()
        except Exception as e:
            print('(could not reach the app:', str(e)[:80] + ')')

    def check(self, qid):
        q = self.qs[qid]
        try:
            ip = get_ipython(); ns = dict(ip.user_ns)
        except NameError:
            ns = dict(_sys.modules['__main__'].__dict__)
        results = []
        def check(cond, msg, hint=''):
            ok = bool(cond); results.append((ok, msg, hint)); return ok
        ns['check'] = check; ns['OUT'] = ''
        for mod, alias in (('numpy', 'np'), ('pandas', 'pd')):
            if alias not in ns:
                try: ns[alias] = __import__(mod)
                except Exception: pass
        if 'plt' not in ns:
            try:
                import matplotlib.pyplot as plt; ns['plt'] = plt
            except Exception: pass
        try:
            exec(compile(q['tests'], '<tests>', 'exec'), ns)
        except BaseException as e:
            results.append((False, 'A check could not finish: ' + type(e).__name__ + ': ' + str(e)[:200], 'Did you create every variable name the task asks for? Did you run your answer cell first?'))
        self.tries[qid] = self.tries.get(qid, 0) + 1
        good = bool(results) and all(r[0] for r in results)
        for ok, msg, hint in results:
            print(('  [PASS] ' if ok else '  [FAIL] ') + msg + ('' if ok or not hint else '\n         hint: ' + hint))
        if good:
            self.ok[qid] = True; print(f"\n  Question solved: {q['title']}  (attempt {self.tries[qid]})")
        else:
            print(f"\n  Not yet - fix your answer cell, run it again, then re-run this check. (attempt {self.tries[qid]})")
        self._report(qid, good)
        return good

    def hint(self, qid):
        hs = self.qs[qid].get('hints') or []
        n = min(self.tries.get(qid, 0), len(hs)) if hs else 0
        print(('hint: ' + hs[max(0, n - 1)]) if hs else 'No hint for this one - re-read the question.')

    def solution(self, qid):
        if self.tries.get(qid, 0) < 3 and not self.ok.get(qid):
            print('Try the check at least 3 times first - the struggle is where the learning happens.'); return
        print(self.qs[qid]['solution'])

    def score(self):
        n = len(self.order); s = sum(1 for q in self.order if self.ok.get(q))
        print(f"Solved {s} of {n}.")
        for q in self.order:
            print(('  [x] ' if self.ok.get(q) else '  [ ] ') + q + ' - ' + self.qs[q]['title'] + ('' if self.ok.get(q) else f"  (tries: {self.tries.get(q, 0)})"))

dsq = _DSQ()


dsq.load({
 "module": "python",
 "title": "Python for Data Science",
 "questions": [
  {
   "id": "py-0-1",
   "title": "Store, calculate, show",
   "tests": "check(price == 250 and quantity == 4, 'price and quantity are stored', 'price = 250 and quantity = 4')\ncheck(subtotal == 1000, 'subtotal is price times quantity = 1000', 'use * between the two names')\ncheck(message == 'Subtotal: 1000', 'the message reads Subtotal: 1000', 'f\"Subtotal: {subtotal}\"')",
   "hints": [
    "Blanks are filled in order. A star * multiplies.",
    "Inside the braces of an f-string you put the NAME of the value."
   ],
   "solution": "price = 250\nquantity = 4\nsubtotal = price * quantity\nmessage = f\"Subtotal: {subtotal}\"\nprint(message)"
  },
  {
   "id": "py-0-2",
   "title": "Name the parts of a line",
   "tests": "check(shape == 'store', 'the line stores a result in discount', 'there is a single = sign putting a value into a name')\ncheck(function_used == 'round', 'it calls round()', 'the word in front of the brackets')\ncheck(argument_count == 2, 'round receives two arguments: the number and the decimals', 'count the commas inside the brackets and add one')",
   "hints": [
    "Is a value being put in a name (store), or is a function just being run (call)?",
    "round(a, b) has two arguments, separated by a comma."
   ],
   "solution": "shape = \"store\"\nfunction_used = \"round\"\nargument_count = 2"
  },
  {
   "id": "py-0-3",
   "title": "Fix the four slips",
   "tests": "check(prices == [120, 80, 200], 'the list is closed and correct', 'add the closing ] on line 1')\ncheck(total == 400 and count == 3, 'total and count are computed', 'sum(prices) and len(prices)')\ncheck('totl' not in dir(), 'every name is spelled the same everywhere', 'compare the name in the f-string with the one you stored')",
   "hints": [
    "Count the [ and ] on the first line.",
    "A line that opens a block ends with a colon.",
    "Check the spelling of the name inside the braces."
   ],
   "solution": "prices = [120, 80, 200]\ntotal = sum(prices)\ncount = len(prices)\nif count > 2:\n    print(f\"{count} items, total {total}\")"
  },
  {
   "id": "py-0-4",
   "title": "Pick the right container",
   "tests": "check(weekdays == ['Mon', 'Tue', 'Wed'], 'a list of three weekday names', 'weekdays = [\"Mon\", \"Tue\", \"Wed\"]')\ncheck(profile == {'name': 'Asha', 'score': 87}, 'a dictionary with name and score', '{\"name\": \"Asha\", \"score\": 87}')\ncheck(second_day == 'Tue', 'the second item is at position 1', 'counting starts at 0')",
   "hints": [
    "Square brackets make a list, curly brackets make a dictionary with label: value.",
    "Positions start at 0, so the second item is position 1."
   ],
   "solution": "weekdays = [\"Mon\", \"Tue\", \"Wed\"]\nprofile = {\"name\": \"Asha\", \"score\": 87}\nsecond_day = weekdays[1]"
  },
  {
   "id": "py-1-1",
   "title": "A bill with GST",
   "tests": "check(abs(total - 118) < 1e-9, 'total = 118', 'price * (1 + gst_rate)')\ncheck(message == 'Total: 118.00', 'message is \"Total: 118.00\"', 'f\"Total: {total:.2f}\"')",
   "hints": [
    "1 + rate is the multiplier; {x:.2f} prints two decimals."
   ],
   "solution": "price = 100\ngst_rate = 0.18\ntotal = price * (1 + gst_rate)\nmessage = f\"Total: {total:.2f}\"\nprint(message)"
  },
  {
   "id": "py-1-2",
   "title": "Divide and remainder",
   "tests": "check(boxes == 7, 'boxes = 7', 'sweets // 6')\ncheck(left == 5, 'left = 5', 'sweets % 6')\ncheck(all_packed is False, 'all_packed is False', 'left == 0')",
   "hints": [
    "// is floor division, % is the remainder."
   ],
   "solution": "sweets = 47\nboxes = sweets // 6\nleft = sweets % 6\nall_packed = left == 0\nprint(boxes, left, all_packed)"
  },
  {
   "id": "py-2-1",
   "title": "Slice and index",
   "tests": "check(first3 == [72, 88, 95], 'first3 = the first three marks', 'marks[:3]')\ncheck(last == 90, 'last = 90', 'marks[-1]')\ncheck(every2 == [72, 95, 79, 90], 'every2 = every second mark', 'marks[::2]')\ncheck(rev == [90, 55, 79, 61, 95, 88, 72], 'rev = reversed', 'marks[::-1]')",
   "hints": [
    "A negative index counts from the end; a negative step walks backwards."
   ],
   "solution": "marks = [72, 88, 95, 61, 79, 55, 90]\nfirst3 = marks[:3]\nlast = marks[-1]\nevery2 = marks[::2]\nrev = marks[::-1]\nprint(first3, last, every2, rev)"
  },
  {
   "id": "py-2-2",
   "title": "Count with a dict",
   "tests": "check(counts == {'data': 3, 'model': 2, 'plot': 1}, 'counts = {data: 3, model: 2, plot: 1}', 'counts[w] = counts.get(w, 0) + 1')\ncheck(most_common == 'data', 'most_common = data', 'max(counts, key=counts.get)')",
   "hints": [
    "counts.get(w, 0) returns 0 for a new word.",
    "max(counts, key=counts.get) returns the key with the biggest value."
   ],
   "solution": "words = [\"data\", \"model\", \"data\", \"plot\", \"model\", \"data\"]\ncounts = {}\nfor w in words:\n    counts[w] = counts.get(w, 0) + 1\nmost_common = max(counts, key=counts.get)\nprint(counts, most_common)"
  },
  {
   "id": "py-3-1",
   "title": "Filter and transform",
   "tests": "check(evens == [2, 4, 6, 8], 'evens = [2, 4, 6, 8]', 'n % 2 == 0')\ncheck(squares_of_odds == [1, 9, 25, 49], 'squares_of_odds = [1, 9, 25, 49]', 'n % 2 != 0')\ncheck(labels == ['small'] * 5 + ['big'] * 3, 'labels are right', '\"big\" if n > 5 else \"small\"')",
   "hints": [
    "Odd numbers have a remainder that is not zero."
   ],
   "solution": "nums = [1, 2, 3, 4, 5, 6, 7, 8]\nevens = [n for n in nums if n % 2 == 0]\nsquares_of_odds = [n ** 2 for n in nums if n % 2 != 0]\nlabels = [\"big\" if n > 5 else \"small\" for n in nums]\nprint(evens, squares_of_odds, labels)"
  },
  {
   "id": "py-3-2",
   "title": "FizzBuzz, the data way",
   "tests": "check(out[:5] == [1, 2, 'Fizz', 4, 'Buzz'] and out[14] == 'FizzBuzz', 'the first five and the last entry are right', 'if / elif / elif / else')\ncheck(len(out) == 15 and out.count('Buzz') == 2, 'there are 15 entries and 2 Buzz', 'range(1, 16)')\ncheck(n_fizz == 4, 'n_fizz = 4', 'out.count(\"Fizz\")')",
   "hints": [
    "Check the most specific case (divisible by 15) first."
   ],
   "solution": "out = []\nfor n in range(1, 16):\n    if n % 15 == 0:\n        out.append(\"FizzBuzz\")\n    elif n % 3 == 0:\n        out.append(\"Fizz\")\n    elif n % 5 == 0:\n        out.append(\"Buzz\")\n    else:\n        out.append(n)\nn_fizz = out.count(\"Fizz\")\nprint(out, n_fizz)"
  },
  {
   "id": "py-4-1",
   "title": "A reusable function",
   "tests": "check(summary([4, 8, 15, 16, 23, 42]) == {'min': 4, 'max': 42, 'mean': 18.0}, 'summary works on the first list', 'return {\"min\": ..., \"max\": ..., \"mean\": ...}')\ncheck(summary([1, 2], ndigits=3) == {'min': 1, 'max': 2, 'mean': 1.5}, 'the ndigits argument is respected', 'round(..., ndigits)')\ncheck(summary([5])['mean'] == 5.0, 'one value works too', 'sum / len')",
   "hints": [
    "return a dict literal with the three keys.",
    "round(x, ndigits)"
   ],
   "solution": "def summary(values, ndigits=1):\n    return {\"min\": min(values), \"max\": max(values), \"mean\": round(sum(values) / len(values), ndigits)}\n\nprint(summary([4, 8, 15, 16, 23, 42]))\nprint(summary([1, 2], ndigits=3))"
  },
  {
   "id": "py-4-2",
   "title": "Parse a CSV, safely",
   "tests": "check(scores == {'Asha': 88.0, 'Chen': 95.0, 'Dara': 61.5}, 'scores holds the three valid rows as floats', 'try: float(...) except ValueError')\ncheck(ranking == ['Chen', 'Asha', 'Dara'], 'ranking = Chen, Asha, Dara', 'sorted(scores, key=scores.get, reverse=True)')",
   "hints": [
    "float(\"seventy\") raises ValueError - catch it.",
    "sorted(scores, key=scores.get, reverse=True)"
   ],
   "solution": "import csv\nimport io\n\ntext = \"\"\"name,score\nAsha,88\nBen,seventy\nChen,95\nDara,61.5\n\"\"\"\nscores = {}\nfor row in csv.DictReader(io.StringIO(text)):\n    try:\n        scores[row[\"name\"]] = float(row[\"score\"])\n    except ValueError:\n        pass\nranking = sorted(scores, key=scores.get, reverse=True)\nprint(scores, ranking)"
  },
  {
   "id": "py-4-1",
   "title": "A price with GST and discount",
   "tests": "check(final_price(100) == 118.0, '100 with default GST is 118.0', 'price * (1 + gst)')\ncheck(final_price(200, discount=0.1) == 212.4, 'a 10% discount first: 180, then GST: 212.4', 'discount comes off before GST')\ncheck(final_price(100, gst=0.05) == 105.0, 'another GST rate', 'the parameter gst can be changed')",
   "hints": [
    "Two steps: apply the discount, then multiply by (1 + gst).",
    "Use return, not print."
   ],
   "solution": "def final_price(price, gst=0.18, discount=0.0):\n    after_discount = price * (1 - discount)\n    return round(after_discount * (1 + gst), 2)\n\nprint(final_price(100))"
  },
  {
   "id": "py-4-2",
   "title": "Summarise a list",
   "tests": "s = summary([70, 85, 90])\ncheck(s == {'count': 3, 'total': 245, 'average': 81.67, 'best': 90}, 'count, total, average and best', 'len, sum, round(..., 2) and max')\ncheck(summary([]) == {'count': 0}, 'an empty list gives just the count', 'check for the empty list first')\ncheck(summary([5])['average'] == 5.0, 'one value: average equals the value', 'divide total by count')",
   "hints": [
    "Handle the empty list first: if not values: return ...",
    "len, sum and max do most of the work."
   ],
   "solution": "def summary(values):\n    if not values:\n        return {\"count\": 0}\n    return {\"count\": len(values), \"total\": sum(values), \"average\": round(sum(values) / len(values), 2), \"best\": max(values)}\n\nprint(summary([70, 85, 90]))"
  },
  {
   "id": "py-5-1",
   "title": "Save and load JSON",
   "tests": "save_scores('t1.json', [1, 2, 3])\ncheck(load_scores('t1.json') == [1, 2, 3], 'what you save is what you load', 'json.dumps then json.loads')\ncheck(load_scores('does_not_exist.json') == [], 'a missing file gives an empty list', 'except FileNotFoundError: return []')\nimport os\ncheck(os.path.exists('t1.json'), 'the file really exists after saving', 'open(path, \"w\")')",
   "hints": [
    "with open(path, \"w\") as f: then f.write(json.dumps(...)).",
    "Catch FileNotFoundError around the read."
   ],
   "solution": "import json\n\ndef save_scores(path, scores):\n    with open(path, \"w\") as f:\n        f.write(json.dumps(scores))\n\ndef load_scores(path):\n    try:\n        with open(path) as f:\n            return json.loads(f.read())\n    except FileNotFoundError:\n        return []\n\nsave_scores(\"scores.json\", [70, 85, 90])\nprint(load_scores(\"scores.json\"), load_scores(\"nothing.json\"))"
  },
  {
   "id": "py-5-2",
   "title": "Clean a column without crashing",
   "tests": "check(to_numbers(['4', 'x', '5.5', '']) == [4.0, 5.5], 'only the convertible entries are kept', 'try float(item), skip on ValueError')\ncheck(average(['4', 'x', '5.5', '']) == 4.75, 'the mean of 4 and 5.5 is 4.75', 'sum / len of the good numbers')\ncheck(average(['a', 'b']) is None, 'no good values gives None', 'check for an empty list first')",
   "hints": [
    "try: float(item) except ValueError: skip.",
    "pass means \"do nothing\"."
   ],
   "solution": "def to_numbers(raw):\n    good = []\n    for item in raw:\n        try:\n            good.append(float(item))\n        except ValueError:\n            pass\n    return good\n\ndef average(raw):\n    nums = to_numbers(raw)\n    if not nums:\n        return None\n    return round(sum(nums) / len(nums), 2)\n\nprint(to_numbers([\"4\", \"x\", \"5.5\", \"\"]), average([\"4\", \"x\", \"5.5\", \"\"]))"
  }
 ]
})


## 1. Store, calculate, show  `py-0-1`
*(from the lesson: What is code? Read your first lines)*

Store the values `price = 250` and `quantity = 4`. Calculate `subtotal` as price times quantity. Store `message` as the text `"Subtotal: 1000"` built with an f-string (`f"Subtotal: {subtotal}"`) and print it.

In [ ]:
price = ___
quantity = ___
subtotal = price ___ quantity
message = f"Subtotal: {___}"
print(message)

In [ ]:
dsq.check("py-0-1")

## 2. Name the parts of a line  `py-0-2`
*(from the lesson: What is code? Read your first lines)*

Read this line aloud: `discount = round(price * 0.1, 2)`. Answer by setting variables: `shape` to `"store"`, `function_used` to the name of the function it calls (text), and `argument_count` to how many arguments that function receives (a number; the comma separates them).

In [ ]:
shape = ___            # "store", "call" or "decide"
function_used = ___    # text
argument_count = ___   # a number

In [ ]:
dsq.check("py-0-2")

## 3. Fix the four slips  `py-0-3`
*(from the lesson: The punctuation of Python)*

This code has four small mistakes (a missing closing bracket, a missing colon, wrong quotes, and a name typed wrongly). Fix them so it prints `3 items, total 400`.

In [ ]:
prices = [120, 80, 200
total = sum(prices)
count = len(prices)
if count > 2
    print(f"{count} items, total {totl}")

In [ ]:
dsq.check("py-0-3")

## 4. Pick the right container  `py-0-4`
*(from the lesson: The punctuation of Python)*

Build three values by shape. `weekdays`: a **list** of the text values `"Mon"`, `"Tue"`, `"Wed"`. `profile`: a **dictionary** with the labels `"name"` (value `"Asha"`) and `"score"` (value `87`). `second_day`: the second item of `weekdays`, picked by position.

In [ ]:
weekdays = ___
profile = ___
second_day = weekdays[___]

In [ ]:
dsq.check("py-0-4")

## 5. A bill with GST  `py-1-1`
*(from the lesson: Values, variables and f-strings)*

Fill the blanks: `total` is price plus 18% GST; `message` is an f-string like `"Total: 118.00"` (two decimals).

In [ ]:
price = 100
gst_rate = 0.18
total = price * (1 + ___)
message = f"Total: {total:.___f}"
print(message)

In [ ]:
dsq.check("py-1-1")

## 6. Divide and remainder  `py-1-2`
*(from the lesson: Values, variables and f-strings)*

You have 47 sweets to pack into boxes of 6. Store the number of **full boxes** in `boxes`, the sweets **left over** in `left`, and a boolean `all_packed` that is True only when nothing is left over.

In [ ]:
sweets = 47
boxes = ...
left = ...
all_packed = ...
print(boxes, left, all_packed)

In [ ]:
dsq.check("py-1-2")

## 7. Slice and index  `py-2-1`
*(from the lesson: Lists, dicts and slicing)*

Fill the blanks to get: the first three marks, the last mark, every second mark, and the marks reversed.

In [ ]:
marks = [72, 88, 95, 61, 79, 55, 90]
first3 = marks[:___]
last = marks[___]
every2 = marks[::___]
rev = marks[::___]
print(first3, last, every2, rev)

In [ ]:
dsq.check("py-2-1")

## 8. Count with a dict  `py-2-2`
*(from the lesson: Lists, dicts and slicing)*

Count how many times each word appears in `words` and store it in the dict `counts` (no imports). Then `most_common` is the word with the highest count.

In [ ]:
words = ["data", "model", "data", "plot", "model", "data"]
counts = {}
for w in words:
    ...
most_common = ...
print(counts, most_common)

In [ ]:
dsq.check("py-2-2")

## 9. Filter and transform  `py-3-1`
*(from the lesson: Decisions, loops and comprehensions)*

Use list comprehensions: `evens` (the even numbers), `squares_of_odds` (the squares of the odd numbers) and `labels` (`"big"` if the number is over 5 else `"small"`, for each number).

In [ ]:
nums = [1, 2, 3, 4, 5, 6, 7, 8]
evens = [n for n in nums if n % 2 == ___]
squares_of_odds = [n ** 2 for n in nums if n % 2 ___ 0]
labels = ["big" if n > 5 else "small" for n in ___]
print(evens, squares_of_odds, labels)

In [ ]:
dsq.check("py-3-1")

## 10. FizzBuzz, the data way  `py-3-2`
*(from the lesson: Decisions, loops and comprehensions)*

Build the list `out` for the numbers 1 to 15: `"FizzBuzz"` if divisible by 3 and 5, `"Fizz"` if by 3, `"Buzz"` if by 5, otherwise the number itself. Also set `n_fizz` to how many entries are exactly `"Fizz"`.

In [ ]:
out = []
for n in range(1, 16):
    ...
n_fizz = ...
print(out, n_fizz)

In [ ]:
dsq.check("py-3-2")

## 11. A reusable function  `py-4-1`
*(from the lesson: Functions, files and errors)*

Write a function `summary(values, ndigits=1)` that returns a dictionary with keys `"min"`, `"max"` and `"mean"` (the mean rounded to `ndigits`).

In [ ]:
def summary(values, ndigits=1):
    ...

print(summary([4, 8, 15, 16, 23, 42]))
print(summary([1, 2], ndigits=3))

In [ ]:
dsq.check("py-4-1")

## 12. Parse a CSV, safely  `py-4-2`
*(from the lesson: Functions, files and errors)*

Read `text` with `csv.DictReader`. Build `scores`: a dict name -> score (float) for the rows whose score can be converted, **skipping** bad rows with try/except. Then `ranking`: the names sorted by score, highest first.

In [ ]:
import csv
import io

text = """name,score
Asha,88
Ben,seventy
Chen,95
Dara,61.5
"""
scores = {}
ranking = []
...
print(scores, ranking)

In [ ]:
dsq.check("py-4-2")

## 13. A price with GST and discount  `py-4-1`
*(from the lesson: Write your own function)*

Write `final_price(price, gst=0.18, discount=0.0)`: take the discount off first (`price * (1 - discount)`), then add GST, and **return** the result rounded to 2 decimals.

In [ ]:
def final_price(price, gst=0.18, discount=0.0):
    ...

print(final_price(100))

In [ ]:
dsq.check("py-4-1")

## 14. Summarise a list  `py-4-2`
*(from the lesson: Write your own function)*

Write `summary(values)` that **returns** a dictionary with the keys `"count"`, `"total"`, `"average"` (rounded to 2 decimals) and `"best"` (the largest value). For an empty list return `{"count": 0}`.

In [ ]:
def summary(values):
    ...

print(summary([70, 85, 90]))

In [ ]:
dsq.check("py-4-2")

## 15. Save and load JSON  `py-5-1`
*(from the lesson: Files, errors and imports)*

Write `save_scores(path, scores)` that writes the list as JSON text into the file, and `load_scores(path)` that reads it back and returns the list. If the file does not exist, `load_scores` should return an empty list `[]`.

In [ ]:
import json

def save_scores(path, scores):
    ...

def load_scores(path):
    ...

save_scores("scores.json", [70, 85, 90])
print(load_scores("scores.json"), load_scores("nothing.json"))

In [ ]:
dsq.check("py-5-1")

## 16. Clean a column without crashing  `py-5-2`
*(from the lesson: Files, errors and imports)*

Survey answers arrive as text, some of them bad. Write `to_numbers(raw)` that returns a list of floats for the entries that can be converted, skipping the ones that cannot (`float("abc")` raises `ValueError`). Then `average(raw)` returns the mean of the good ones rounded to 2 decimals, or `None` if there are none.

In [ ]:
def to_numbers(raw):
    ...

def average(raw):
    ...

print(to_numbers(["4", "x", "5.5", ""]), average(["4", "x", "5.5", ""]))

In [ ]:
dsq.check("py-5-2")

## Done?

In [ ]:
dsq.score()